# RL Environments

In [43]:
import random
import torch

def setup() -> tuple[torch.Generator, int]:
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        torch.set_default_device(device)
        generator = torch.Generator(device=device)
        print("MPS device found!")
    else:
        device = torch.device("cpu")
        generator = torch.Generator(device=device)
        print("Defaulting to CPU!")

    SEED = random.SystemRandom().randint(0, 2**32 - 1)
    torch.manual_seed(SEED)
    print(f"Random seed: {SEED}")
    return generator, SEED


generator, SEED = setup()

MPS device found!
Random seed: 3392617935


# Raw Environment

In [44]:
import gymnasium as gym

env = gym.make("CartPole-v1")

observation, info = env.reset(seed=SEED)

for _ in range(10):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)

    print(observation)

    if terminated or truncated:
        observation, info = env.reset()
env.close()


[-0.00873226  0.16909447 -0.02565453 -0.28285903]
[-0.00535037 -0.02565235 -0.03131171  0.00162348]
[-0.00586341 -0.22031158 -0.03127924  0.28426507]
[-0.01026964 -0.02475779 -0.02559394 -0.01811674]
[-0.0107648   0.17072168 -0.02595628 -0.3187637 ]
[-0.00735037 -0.02402116 -0.03233155 -0.03437813]
[-0.00783079  0.17154916 -0.03301911 -0.3370841 ]
[-0.00439981  0.36712506 -0.0397608  -0.6399939 ]
[ 0.00294269  0.17257933 -0.05256068 -0.36009258]
[ 0.00639428 -0.0217576  -0.05976253 -0.08443546]


## TorchRL Environment

In [45]:
from torchrl.envs import GymEnv

env = GymEnv("CartPole-v1")

td = env.reset(seed=SEED)
print(td)
      
for _ in range(10):
    # Random action
    td = env.rand_step(td)

    print(td["next", "observation"])

    # Check whether the episode ended
    if td["next", "done"]:
        td = env.reset()

print(td)
env.close()

TensorDict(
    fields={
        done: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=True),
        observation: Tensor(shape=torch.Size([4]), device=mps:0, dtype=torch.float32, is_shared=True),
        terminated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=True),
        truncated: Tensor(shape=torch.Size([1]), device=mps:0, dtype=torch.bool, is_shared=True)},
    batch_size=torch.Size([]),
    device=None,
    is_shared=False)
tensor([-0.0087, -0.2211, -0.0257,  0.3023], device='mps:0')
tensor([-0.0132, -0.4159, -0.0196,  0.5868], device='mps:0')
tensor([-0.0215, -0.6107, -0.0079,  0.8732], device='mps:0')
tensor([-0.0337, -0.4155,  0.0096,  0.5781], device='mps:0')
tensor([-0.0420, -0.6107,  0.0212,  0.8737], device='mps:0')
tensor([-0.0542, -0.8061,  0.0386,  1.1730], device='mps:0')
tensor([-0.0703, -0.6115,  0.0621,  0.8927], device='mps:0')
tensor([-0.0826, -0.4173,  0.0799,  0.6201], device='mps:0')
tensor([-0.0909, -0.2234

# Torch RL TensorDictReplayBuffer

In [46]:
from torchrl.envs import GymEnv
from torchrl.data import (
    TensorDictReplayBuffer,
    LazyTensorStorage,
)

sample_batch_size = 16
episode_length = 200
storage_mem_max = episode_length

env = GymEnv("CartPole-v1")
print(f"Environment: {env}")
print(f"Environment batch size: {env.batch_size}")
print(f"Environment observation spec: {env.observation_spec}")
print(f"Environment action spec: {env.action_spec}")
print(f"Environment state spec: {env.state_spec}")

td = env.reset()
replay_buffer = TensorDictReplayBuffer(
    storage=LazyTensorStorage(storage_mem_max),
    batch_size=sample_batch_size,
)

for _ in range(episode_length):

    # Random action for this example
    action = env.action_spec.rand()

    # Put action into current TensorDict
    td["action"] = action

    # Step environment
    td = env.step(td)

    # Store the transition
    replay_buffer.add(td)

    # Continue from next state
    td = td["next"]

    # Reset when episode ends
    if td["done"].item():
        td = env.reset()



batch = replay_buffer.sample(batch_size=10)

print(batch)
print(batch.batch_size)

print("obs:", batch["observation"].shape)
print("action:", batch["action"].shape)
print("reward:", batch["next", "reward"].shape)
print("next obs:", batch["next", "observation"].shape)

Environment: GymEnv(env=CartPole-v1, batch_size=torch.Size([]), device=None)
Environment batch size: torch.Size([])
Environment observation spec: Composite(
    observation: BoundedContinuous(
        shape=torch.Size([4]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([4]), device=mps:0, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([4]), device=mps:0, dtype=torch.float32, contiguous=True)),
        device=mps:0,
        dtype=torch.float32,
        domain=continuous),
    device=None,
    shape=torch.Size([]),
    data_cls=None)
Environment action spec: OneHot(
    shape=torch.Size([2]),
    space=CategoricalBox(n=2),
    device=mps:0,
    dtype=torch.int64,
    domain=discrete)
Environment state spec: Composite(device=None, shape=torch.Size([]), data_cls=None)
2026-10-03 23:31:48,074 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([200]) shape [END]
TensorDict(
    fields={
        action: Tensor(shape=torch.S

/Users/ronnie/Local Documents/Non iCloud/Repos/ronniestafford/ai-ml-research/.venv/lib/python3.14/site-packages/torchrl/data/replay_buffers/replay_buffers/base.py:2314: UserWarning: Got conflicting batch_sizes in constructor (16) and `sample` (10). Refer to the ReplayBuffer documentation for a proper usage of the batch-size arguments. The batch-size provided to the sample method will prevail.
  warnings.warn(
